# Task 5: Interpret Results

Summarize the performance of all models and visualize decision boundaries (using PCA for dimensionality reduction) to provide biological insights into cancer classification.

**Goal**: Synthesize findings and visualize model behavior.

## 1. Initialize Project Environment

In [1]:
import logging
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.svm import SVC

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)

print(f"Python {sys.version}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]


## 2. Define Configuration Parameters

In [2]:
@dataclass
class TaskConfig:
    handle: str
    artifacts_dir: Path = Path("artifacts")
    random_state: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["artifacts_dir"] = str(info["artifacts_dir"])
        return info


CONFIG = TaskConfig(handle="rbals")
CONFIG.describe()

{'handle': 'rbals', 'artifacts_dir': 'artifacts', 'random_state': 42}

## 3. Implement Core Functionality

In [3]:
def plot_decision_boundaries(config: TaskConfig):
    # Load processed data
    X_train = pd.read_csv(config.artifacts_dir / "task1_X_train_scaled.csv")
    y_train = pd.read_csv(config.artifacts_dir / "task1_y_train.csv").values.flatten()

    # Load optimized params
    optimized_params = pd.read_csv(
        config.artifacts_dir / "task4_optimized_svm_params.csv"
    ).iloc[0]

    # PCA to 2D
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_train)

    # Train SVM on 2D space for visualization
    svm = SVC(
        C=optimized_params["C"],
        gamma=optimized_params["gamma"],
        kernel="rbf",
        random_state=config.random_state,
    )
    svm.fit(X_pca, y_train)

    # Create meshgrid
    h = 0.02
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    # Predict over meshgrid
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.scatter(
        X_pca[:, 0], X_pca[:, 1], c=y_train, cmap=plt.cm.coolwarm, edgecolors="k"
    )
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title("SVM Decision Boundary (PCA-Reduced Space)")

    out_img = config.artifacts_dir / "task5_decision_boundary.png"
    plt.savefig(out_img)
    plt.close()

    logging.info(f"Decision boundary plot saved to {out_img}")

    return out_img


DECISION_BOUNDARY_PLOT = plot_decision_boundaries(CONFIG)

2026-02-01 16:32:12,970 | INFO | Decision boundary plot saved to artifacts/task5_decision_boundary.png


## 4. Validate with Unit Tests

In [4]:
assert DECISION_BOUNDARY_PLOT.exists(), "Plot failed to generate"
print("[OK] Validation passed.")

[OK] Validation passed.


## 5. Export Results

In [5]:
EXPORT_DIR = CONFIG.artifacts_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Synthesis report placeholder (Quantitative results only)
synthesis = {
    "num_features": pd.read_csv(EXPORT_DIR / "task1_X_train_scaled.csv").shape[1],
    "best_svm_score": pd.read_csv(EXPORT_DIR / "task4_svm_cv_results.csv")[
        "mean_test_score"
    ].max(),
    "logreg_accuracy": pd.read_csv(
        EXPORT_DIR / "task2_logistic_regression_metrics.csv"
    )["accuracy"].iloc[0],
}

synthesis_df = pd.DataFrame([synthesis])
synthesis_out = EXPORT_DIR / "task5_final_synthesis.csv"
synthesis_df.to_csv(synthesis_out, index=False)

print(f"[OK] Final synthesis results saved to artifacts/")

[OK] Final synthesis results saved to artifacts/
